In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

## Read the files

In [2]:
def read_ragged_as_arrays(path, sep=",", dtype=float):
    rows = []
    with Path(path).open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = [v.strip() for v in line.split(sep) if v.strip() != ""]
            rows.append(np.asarray(vals, dtype=dtype))
    return rows

In [4]:
FilePath = "./GeneratedData/"
Ci_ragged   = read_ragged_as_arrays(FilePath + "n3data_Ci.txt",   dtype=float)
Anet_ragged = read_ragged_as_arrays(FilePath + "n3data_Anet.txt", dtype=float)
ID_ragged   = read_ragged_as_arrays(FilePath + "n3data_ID.txt",   dtype=int)
Params_ragged  = read_ragged_as_arrays(FilePath + "n3data_Params.txt",   dtype=float)

ValueError: invalid literal for int() with base 10: '1.0'

## Save data into excel

In [6]:
n = len(Ci_ragged)

# =========================
# 1) Points sheet (GNN-ready long format)
# =========================
rows = []
for curve_id in range(n):
    Ci   = np.asarray(Ci_ragged[curve_id], dtype=float)
    Anet = np.asarray(Anet_ragged[curve_id], dtype=float)
    ID   = np.asarray(ID_ragged[curve_id], dtype=int)

    m = min(len(Ci), len(Anet), len(ID))
    if m == 0:
        continue

    # Keep node ordering stable (important for reproducibility)
    # Optionally sort by Ci if your curves can be out-of-order:
    # idx = np.argsort(Ci[:m])
    # Ci, Anet, ID = Ci[idx], Anet[idx], ID[idx]

    for point_id in range(m):
        rows.append([curve_id, point_id, float(Ci[point_id]), float(Anet[point_id]), int(ID[point_id])])

df_points = pd.DataFrame(rows, columns=["curve_id", "point_id", "Ci", "Anet", "ID"])

# =========================
# 2) Params sheet (fixed schema, named columns)
# =========================
param_cols = ["Vcmax25", "J25", "TPU25", "Rd25", "gm25", "Vcmax", "J", "TPU", "Rd", "gm","Tleaf"]
expected_p = len(param_cols)

param_rows = []
for curve_id in range(n):
    p = np.asarray(Params_ragged[curve_id], dtype=float).reshape(-1)

    # enforce fixed length; if your file has more/less, handle it explicitly
    if len(p) < expected_p:
        padded = np.full(expected_p, np.nan, dtype=float)
        padded[:len(p)] = p
        p = padded
    elif len(p) > expected_p:
        p = p[:expected_p]  # keep only the first 6 in the specified order

    param_rows.append([curve_id, *[float(v) if not np.isnan(v) else np.nan for v in p]])

df_params = pd.DataFrame(param_rows, columns=["curve_id", *param_cols])

# =========================
# 3) Write Excel files
# =========================
points_xlsx = FilePath + "nACi_points.xlsx"
params_xlsx = FilePath + "nACi_params.xlsx"

with pd.ExcelWriter(points_xlsx, engine="openpyxl") as writer:
    df_points.to_excel(writer, sheet_name="points", index=False)

with pd.ExcelWriter(params_xlsx, engine="openpyxl") as writer:
    df_params.to_excel(writer, sheet_name="params", index=False)

print("Wrote:")
print(" -", points_xlsx)
print(" -", params_xlsx)
print("Points rows:", len(df_points), "| Curves:", df_points["curve_id"].nunique())
print("Params rows:", len(df_params))

Wrote:
 - /content/drive/My Drive/Colab Notebooks/ACi_Data/nACi_points.xlsx
 - /content/drive/My Drive/Colab Notebooks/ACi_Data/nACi_params.xlsx
Points rows: 62785 | Curves: 5449
Params rows: 5449
